# Leaderboards

The same three Azure deployments, four questions with known answers, ranked by correct answers and then by estimated cost.

Public tables answer different questions. Artificial Analysis puts an intelligence index next to price and speed. The Hugging Face open LLM leaderboard scores open weights on a fixed set. Vellum writes up new releases. SEAL is a private exam written so the model has not memorized it. LiveBench changes the questions so a leaked set goes stale. LM Arena is blind votes, and the rating is Elo.

A single rank mixes three commercial jobs. Automation finishes the task and a person checks exceptions. Augmentation drafts while a person stays in the loop. Agentic work lets the model pick tools and take several steps.

The prices below are example USD per million tokens, input then output. Replace them with the rates on your Azure resource.


## Client

In [ ]:
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from openai import AzureOpenAI

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / ".git").exists()), cwd)
load_dotenv(repo_root / ".env", override=True)

client = AzureOpenAI(
    api_key=os.getenv("GPT_KEY"),
    api_version=os.getenv("GPT_API_VERSION"),
    azure_endpoint=os.getenv("GPT_ENDPOINT"),
)
azure_model = os.getenv("GPT_MODEL", "gpt-4o")
deployments = []
for name in (azure_model, "gpt-4o-mini", "gpt-4.1"):
    if name not in deployments:
        deployments.append(name)

usd_per_million = {
    "gpt-4o": (2.50, 10.00),
    "gpt-4o-mini": (0.15, 0.60),
    "gpt-4.1": (2.00, 8.00),
}


## Questions

The expected answer has to show up as its own word. `408` matches. `113` does not match `13`.

In [ ]:
questions = [
    ("What is 17 * 24?", "408"),
    ("What is the capital of Sri Lanka?", "colombo"),
    ("What is the next prime after 11?", "13"),
    ("How many minutes are there from 3:40 to 5:05?", "85"),
]


def matches(reply, expected):
    tokens = []
    for word in reply.lower().replace(",", " ").split():
        tokens.append("".join(ch for ch in word if ch.isalnum()))
    return expected in tokens


def dollars(deployment, usage):
    rates = usd_per_million.get(deployment)
    if not rates or not usage:
        return 0.0
    in_rate, out_rate = rates
    return (usage.prompt_tokens * in_rate + usage.completion_tokens * out_rate) / 1_000_000


## Run

In [ ]:
def run(deployment):
    correct = 0
    spent = 0.0
    start = time.perf_counter()
    for question, expected in questions:
        response = client.chat.completions.create(
            model=deployment,
            messages=[
                {"role": "system", "content": "Reply with the answer only."},
                {"role": "user", "content": question},
            ],
            timeout=120.0,
        )
        reply = response.choices[0].message.content or ""
        ok = matches(reply, expected)
        correct += ok
        spent += dollars(deployment, response.usage)
        mark = "ok" if ok else f"wanted {expected}"
        print(f"{deployment}  {question}  {reply.strip()}  {mark}")
    seconds = time.perf_counter() - start
    print()
    return {"model": deployment, "correct": correct, "seconds": seconds, "dollars": spent}


## Table

More correct answers rank higher. A tie goes to the cheaper run.

In [ ]:
rows = [run(name) for name in deployments]
rows.sort(key=lambda row: (-row["correct"], row["dollars"], row["model"]))

print(f"{'Rank':<5} {'Model':<16} {'Score':<8} {'Seconds':<10} Cost")
for rank, row in enumerate(rows, start=1):
    print(
        f"{rank:<5} {row['model']:<16} {row['correct']}/{len(questions):<6} "
        f"{row['seconds']:<10.1f} ${row['dollars']:.4f}"
    )
